In [ ]:
# 라이브러리 import
import getpass
import time
import pandas as pd
import plotly.graph_objects as go
import requests

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

from tabulate import tabulate

# '강의명'을 통해, SNULIFE 강의실에서 해당 강의를 검색하고 족보 정보('이름', '연도/학기', '링크')를 반환함
def find_lecture_reference(title):
    '''
    title : 강의명 (str)
    '''

    # 1. 크롬 드라이버 경로 설정
    chromedriver_path = './chromedriver.exe'
    service = Service(executable_path=chromedriver_path)

    # 2.1. 헤드리스 옵션 설정
    chrome_options = Options()
    chrome_options.add_argument('--headless') # 헤드리스 옵션 추가
    chrome_options.add_argument('--no-sandbox') # 안정성 장치 (1)
    chrome_options.add_argument('--disable-dev-shm-usage') # 안정성 장치 (2)

    # 2.2. chrome 웹 브라우저 열기
    driver = webdriver.Chrome(service=service)
    driver.maximize_window() # 창 최대화
    
    # LOG
    print("STEP 1. Chrome 브라우저 실행 완료.")

    # 3. SNULIFE 강의실 url 접속
    driver.get("https://www.snulife.com/lecture")

    # LOG
    print("STEP 2. SNULIFE 강의실 접속 완료.")

    try:
        wait = WebDriverWait(driver, 2) # 2초 대기 설정

        # 4. 로그인
        while True:
            login_box_xpath = '//*[@id="__next"]/div/div[2]/div[2]/div[1]/div[2]/div[1]/div/form/div[1]/div'

            try:
                # 4.1. id/pw 요소 찾기
                id_box = wait.until(EC.element_to_be_clickable((By.XPATH, login_box_xpath+'/input[1]')))
                pw_box = driver.find_element(By.XPATH, login_box_xpath+'/input[2]')

                # 4.2. id/pw 요소 비우기 -> id/pw 받기 -> id/pw 입력 -> ENTER
                id_box.clear()
                id = getpass.getpass("ID를 입력해주세요.")
                id_box.send_keys(id)

                pw_box.clear()
                pw = getpass.getpass("PW를 입력해주세요.")
                pw_box.send_keys(pw) 

                pw_box.send_keys(Keys.ENTER)

                # 4.5. '로그인 실패' 팝업 확인
                wait.until(EC.visibility_of_element_located((By.XPATH, "//*[contains(text(), '아이디 또는 비밀번호를 다시 확인해주세요.')]")))

                # LOG
                print("로그인에 실패하였습니다. ID와 PW를 다시 입력해주세요.")

                # 4.6. 로그인 실패 -> '확인' 버튼 누른 후, 로그인 다시 시도
                driver.find_element(By.XPATH, '//button[text()="확인"]').click()
                
                continue
            
            # 4.7. 로그인 성공 -> 다음 로직으로 넘어가기
            except TimeoutException:
                # LOG
                print("STEP 3. 로그인 완료.")

                break

        # 5. 강의 검색
        title = title.replace(" ", "").lower() # 띄어쓰기 제거 및 소문자로 변경
        search_box = driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[1]/div/div/div[1]/div/div/input') # '검색 창' 찾기
        search_box.send_keys(title) # '강의명' 입력
        search_box.send_keys(Keys.ENTER)

        wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@id="__next"]/div/div[2]/div')))

        # 6.1. 검색 결과 개수 구하기
        a_xpath = '//*[@id="__next"]/div/div[2]/div/div[1]/div[3]'
        a_elem = driver.find_element(By.XPATH, a_xpath)
        a_elem_num = len(a_elem.find_elements(By.XPATH, './a'))

        # LOG
        print(f"STEP 4. 강의 검색 완료. 총 {a_elem_num}개의 강의를 찾았습니다.")

        # 6.2. 검색 결과 수집 -> list에 저장
        search_results = []

        for i in range(1, a_elem_num+1):
            tmp_xpath = f'//*[@id="__next"]/div/div[2]/div/div[1]/div[3]/a[{i}]'

            tmp_title = driver.find_element(By.XPATH, tmp_xpath+'/div[1]/div[2]').text # 강의명
            tmp_prof_name = driver.find_element(By.XPATH, tmp_xpath+'/div[2]/div[1]/span[1]').text # 교수 이름
            tmp_subject_classification = driver.find_element(By.XPATH, tmp_xpath+'/div[1]/div[1]').text # 교과 구분
            tmp_dep = driver.find_element(By.XPATH, tmp_xpath+'/div[2]/div[1]/span[3]').text # 개설 학과
            tmp_rating = driver.find_element(By.XPATH, tmp_xpath+'/div[1]/div[3]').text # 별점
            tmp_review_num = driver.find_element(By.XPATH, tmp_xpath+'/div[2]/div[2]/span[1]').text[4:] # 강의평 개수
            tmp_reference_num = driver.find_element(By.XPATH, tmp_xpath+'/div[2]/div[2]/span[3]').text[3:] # 족보 개수

            search_results.append([i, tmp_title, tmp_prof_name, tmp_subject_classification, tmp_dep, tmp_rating, tmp_review_num, tmp_reference_num])

        # 6.3. 검색 결과 반환
        if search_results:
            header_values = ['INDEX', '강의명', '교수', '교과 구분', '개설 학과', '별점(10점 만점)', '강의평', '족보'] # column 제목 설정
            search_results = [list(x) for x in zip(*search_results)]

            fig = go.Figure()
            fig.add_trace(
                go.Table(header=dict(values=header_values),
                         cells=dict(values=search_results))
            )
            fig.show()

        else:
            # LOG
            print("ERROR : 조건에 맞는 강의가 존재하지 않습니다.")

            return []
        

        # 7.1. 사용자가 원하는 강의 클릭
        target_index = int(input(F"원하는 강의의 INDEX를 입력해주세요. 1부터 {a_elem_num}까지 숫자 중 하나를 입력해주세요."))
        reference_num = int(search_results[-1][target_index-1]) # 해당 강의의 족보 개수

        # 7.2. 족보가 있는 경우
        if reference_num > 0:
            # LOG
            print(f"STEP 5. 족보 찾기 완료. 총 {reference_num}개의 족보를 찾았습니다.")

            # 7.2.1. 족보 정보 수집을 위한 페이지 이동
            driver.find_element(By.XPATH, f'//*[@id="__next"]/div/div[2]/div/div[1]/div[3]/a[{target_index}]').click()
            wait.until(EC.visibility_of_element_located((By.XPATH, '//*[@id="__next"]/div/div[2]/div[4]/div/button[2]'))).click() # '족보' 버튼이 나타날 때까지 대기 후, 버튼 클릭
            wait.until(lambda driver : driver.find_element(By.XPATH, '//*[@id="__next"]/div/div[2]/div[4]/div/button[2]').get_attribute('class') == 'css-1oi3o5y') # '족보' 버튼이 정상적으로 클릭 완료 될 때까지 대기

            # 7.2.2. 족보 정보 수집 -> list에 저장
            reference_results = []

            for i in range(1, reference_num+1):
                tmp_link = driver.find_element(By.XPATH, f'//*[@id="__next"]/div/div[2]/div[5]/div/div[{i}]/div[4]/a').get_attribute('href') # 파일 링크
                tmp_semester = driver.find_element(By.XPATH, f'//*[@id="__next"]/div/div[2]/div[5]/div/div[{i}]/div[1]/div').text # 연도/학기
                tmp_title = driver.find_element(By.XPATH, f'//*[@id="__next"]/div/div[2]/div[5]/div/div[{i}]/div[2]/button').text # 파일 제목

                reference_results.append([tmp_title, tmp_semester, tmp_link])
        
            return reference_results
        
        else:
            # LOG
            print(f"ERROR : 해당 강의의 족보가 존재하지 않습니다.")

            return []

                    
    except Exception as e:
        print(f"ERROR : {e}")

In [ ]:
search_query = "산업"
ref_results = find_lecture_reference(search_query)

STEP 1. Chrome 브라우저 실행 완료.
STEP 2. SNULIFE 강의실 접속 완료.
STEP 3. 로그인 완료.
STEP 4. 강의 검색 완료. 총 20개의 강의를 찾았습니다.


STEP 5. 족보 찾기 완료. 총 6개의 족보를 찾았습니다.


In [13]:
header_values = ['파일 제목', '연도/학기', '파일 링크'] # column 제목 설정
ref_results1 = [list(x) for x in zip(*ref_results)]

fig = go.Figure()
fig.add_trace(
    go.Table(header=dict(values=header_values),
                cells=dict(values=ref_results1, align='left'),
                columnwidth=[80, 100, 700])
)
fig.show()